# Smoking Status — Tabular NN (FT-Transformer)


## 1) Импорты и конфиг


In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [2]:
CFG = {
    "n_splits": 5,
    "seeds": [42, 2025],
    "epochs": 50,
    "patience": 8,
    "batch_size": 1024,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "d_token": 192,
    "n_heads": 8,
    "n_layers": 5,
    "d_ff": 768,
    "dropout": 0.15,
    "min_delta": 1e-4,
    "num_workers": 2,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

## 2) Загрузка данных


In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

target_col = "smoking" if "smoking" in train.columns else ("target" if "target" in train.columns else None)
if target_col is None:
    raise ValueError("Не найден столбец таргета: ожидался smoking или target")

id_col = "id"
features = [c for c in train.columns if c not in [target_col] and c != id_col]

y = train[target_col].astype(int).values
train_ids = train[id_col].values
test_ids = test[id_col].values

len(features), features[:5]

(22, ['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)'])

## 3) Числовые/категориальные + заполнение пропусков + log1p


In [5]:
full = pd.concat([train[features], test[features]], axis=0, ignore_index=True)

low_card_threshold = 32
cat_cols = []
num_cols = []

for c in features:
    s = full[c]
    if s.dtype == "object":
        cat_cols.append(c)
    else:
        nun = s.nunique(dropna=True)
        if nun <= low_card_threshold:
            cat_cols.append(c)
        else:
            num_cols.append(c)

log1p_cols = [c for c in [
    "triglyceride", "LDL", "HDL", "AST", "ALT", "Gtp",
    "serum creatinine", "Cholesterol", "fasting blood sugar"
] if c in num_cols]

train_num = train[num_cols].copy()
test_num = test[num_cols].copy()

for c in num_cols:
    med = pd.concat([train_num[c], test_num[c]], axis=0).median()
    train_num[c] = train_num[c].fillna(med)
    test_num[c] = test_num[c].fillna(med)

for c in log1p_cols:
    train_num[c] = np.log1p(np.maximum(train_num[c].values, 0))
    test_num[c] = np.log1p(np.maximum(test_num[c].values, 0))

train_cat = train[cat_cols].copy()
test_cat = test[cat_cols].copy()

for c in cat_cols:
    s_all = pd.concat([train_cat[c], test_cat[c]], axis=0)
    mode = s_all.mode(dropna=True)
    fillv = mode.iloc[0] if len(mode) else 0
    train_cat[c] = train_cat[c].fillna(fillv)
    test_cat[c] = test_cat[c].fillna(fillv)

len(num_cols), len(cat_cols), log1p_cols


(12,
 10,
 ['triglyceride',
  'LDL',
  'HDL',
  'AST',
  'ALT',
  'Gtp',
  'Cholesterol',
  'fasting blood sugar'])

## 4) Кодирование категорий


In [6]:
cat_maps = {}
cat_sizes = []

for c in cat_cols:
    s_all = pd.concat([train_cat[c], test_cat[c]], axis=0).astype(str)
    uniq = pd.Index(s_all.unique())
    m = {k: i + 1 for i, k in enumerate(uniq)}
    cat_maps[c] = m
    cat_sizes.append(len(uniq) + 1)

def encode_cats(df_cat):
    arrs = []
    for c in cat_cols:
        m = cat_maps[c]
        a = df_cat[c].astype(str).map(m).fillna(0).astype(np.int64).values
        arrs.append(a)
    if len(arrs) == 0:
        return np.zeros((len(df_cat), 0), dtype=np.int64)
    return np.stack(arrs, axis=1)

X_cat = encode_cats(train_cat)
T_cat = encode_cats(test_cat)

X_num = train_num.values.astype(np.float32)
T_num = test_num.values.astype(np.float32)

X_cat.shape, X_num.shape, T_cat.shape, T_num.shape


((15000, 10), (15000, 12), (10000, 10), (10000, 12))

## 5) Dataset/DataLoader


In [7]:
class TabDS(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.from_numpy(x_num)
        self.x_cat = torch.from_numpy(x_cat)
        self.y = None if y is None else torch.from_numpy(y.astype(np.float32))
    def __len__(self):
        return self.x_num.shape[0]
    def __getitem__(self, i):
        if self.y is None:
            return self.x_num[i], self.x_cat[i]
        return self.x_num[i], self.x_cat[i], self.y[i]


## 6) FT-Transformer


In [8]:
class FFN(nn.Module):
    def __init__(self, d_token, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_token, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_token),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_token, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_token)
        self.ffn = FFN(d_token, d_ff, dropout)
    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(h)
        x = x + self.ffn(self.norm2(x))
        return x

class FTTransformer(nn.Module):
    def __init__(self, n_num, cat_sizes, d_token, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_sizes)

        if self.n_num > 0:
            self.num_w = nn.Parameter(torch.randn(self.n_num, d_token) * 0.02)
            self.num_b = nn.Parameter(torch.zeros(self.n_num, d_token))

        self.cat_embs = nn.ModuleList([nn.Embedding(cs, d_token) for cs in cat_sizes])

        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        self.blocks = nn.ModuleList([TransformerBlock(d_token, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_token)

        self.head = nn.Sequential(
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1),
        )

    def forward(self, x_num, x_cat):
        bs = x_num.shape[0]
        tokens = []

        if self.n_cat > 0:
            for j, emb in enumerate(self.cat_embs):
                tokens.append(emb(x_cat[:, j]).unsqueeze(1))

        if self.n_num > 0:
            xn = x_num.unsqueeze(-1)
            num_tok = xn * self.num_w.unsqueeze(0) + self.num_b.unsqueeze(0)
            tokens.append(num_tok)

        if len(tokens) == 0:
            x = self.cls.expand(bs, -1, -1)
        else:
            x = torch.cat(tokens, dim=1)
            x = torch.cat([self.cls.expand(bs, -1, -1), x], dim=1)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        cls = x[:, 0]
        return self.head(cls).squeeze(1)


## 7) Тренировка фолда


In [9]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps = []
    for batch in loader:
        x_num, x_cat = batch[:2]
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True)
        logits = model(x_num, x_cat)
        ps.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(ps)

def train_fold(xn_tr, xc_tr, y_tr, xn_va, xc_va, y_va, xn_te, xc_te, seed):
    set_seed(seed)

    model = FTTransformer(
        n_num=xn_tr.shape[1],
        cat_sizes=cat_sizes,
        d_token=CFG["d_token"],
        n_heads=CFG["n_heads"],
        n_layers=CFG["n_layers"],
        d_ff=CFG["d_ff"],
        dropout=CFG["dropout"],
    ).to(device)

    tr_ds = TabDS(xn_tr, xc_tr, y_tr)
    va_ds = TabDS(xn_va, xc_va, y_va)
    te_ds = TabDS(xn_te, xc_te, None)

    tr_loader = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True, drop_last=False)
    te_loader = DataLoader(te_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = max(1, len(tr_loader) * 10)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps, pct_start=0.1, div_factor=10.0, final_div_factor=50.0)

    bce = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_auc = -1.0
    best_state = None
    patience = 0

    for epoch in range(CFG["epochs"]):
        model.train()
        for x_num, x_cat, yy in tr_loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            yy = yy.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x_num, x_cat)
                loss = bce(logits, yy)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            if sched.last_epoch < sched.total_steps - 1:
                sched.step()

        p_va = predict_proba(model, va_loader)
        auc = roc_auc_score(y_va, p_va)

        if auc > best_auc + CFG["min_delta"]:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= CFG["patience"]:
                break

    model.load_state_dict(best_state)
    p_va = predict_proba(model, va_loader)
    p_te = predict_proba(model, te_loader)
    return best_auc, p_va, p_te


## 8) K-Fold + ансамбль + submission.csv


In [ ]:
def clip_and_scale_per_fold(x_tr, x_va, x_te, q=0.001):
    if x_tr.shape[1] == 0:
        return x_tr, x_va, x_te

    lo = np.quantile(x_tr, q, axis=0)
    hi = np.quantile(x_tr, 1 - q, axis=0)

    x_tr = np.clip(x_tr, lo, hi)
    x_va = np.clip(x_va, lo, hi)
    x_te = np.clip(x_te, lo, hi)

    mu = x_tr.mean(axis=0)
    sd = x_tr.std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)

    x_tr = (x_tr - mu) / sd
    x_va = (x_va - mu) / sd
    x_te = (x_te - mu) / sd
    return x_tr.astype(np.float32), x_va.astype(np.float32), x_te.astype(np.float32)

oof = np.zeros(len(train), dtype=np.float64)
test_pred = np.zeros(len(test), dtype=np.float64)

skf = StratifiedKFold(n_splits=CFG["n_splits"], shuffle=True, random_state=1337)

fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_num, y), 1):
    xn_tr0, xn_va0 = X_num[tr_idx].copy(), X_num[va_idx].copy()
    xc_tr, xc_va = X_cat[tr_idx].copy(), X_cat[va_idx].copy()
    y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()

    fold_oof = np.zeros(len(va_idx), dtype=np.float64)
    fold_test = np.zeros(len(test), dtype=np.float64)
    fold_seed_scores = []

    for seed in CFG["seeds"]:
        xn_tr, xn_va, xn_te = clip_and_scale_per_fold(xn_tr0.copy(), xn_va0.copy(), T_num.copy(), q=0.001)

        auc, p_va, p_te = train_fold(
            xn_tr, xc_tr, y_tr,
            xn_va, xc_va, y_va,
            xn_te, T_cat, seed
        )

        fold_seed_scores.append(auc)
        fold_oof += p_va / len(CFG["seeds"])
        fold_test += p_te / len(CFG["seeds"])

    oof[va_idx] = fold_oof
    test_pred += fold_test / CFG["n_splits"]
    fold_scores.append((fold, float(np.mean(fold_seed_scores)), float(np.std(fold_seed_scores))))

cv_auc = roc_auc_score(y, oof)
cv_auc, fold_scores


/tmp/ipython-input-1263534693.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipython-input-1263534693.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipython-input-1263534693.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipython-input-1263534693.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipython-input-1263534693.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is d

In [ ]:
sub = pd.DataFrame({id_col: test_ids, "smoking": test_pred.astype(np.float64)})
sub.to_csv("submission.csv", index=False)
sub.head(), sub.shape
